# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [3]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [4]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'24464b1d-acad-4851-b9a7-214a5aa76e96': {'version…

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

1) Zero shot prompting: the model is given a task and asked to perform it without any examples. The model relies solely on its pre-trained knowledge to generate a response.

In [22]:
prompt1 = """
Take a food order from a customer by start saying you are prompt 1 for a fast food restaurant.
Be friendly, ask for the full order, confirm it, and ask for payment.
Menu:
burger, frankfurt, sandwich, fries, salad
drinks: coke, sprite, vichy catalan
"""

2.Few shot

In [20]:
prompt2 = """
You are an OrderBot in a fast food restaurant.

Example conversation:

Customer: Hi
Assistant: Hello! What would you like to order today?

Customer: I want a burger and a coke
Assistant: Great! What size would you like for your burger and drink?

Customer: Medium please
Assistant: Perfect. So that's a medium burger and a medium coke. Anything else?

Customer: No
Assistant: Your total is ready. Please proceed to payment.

Now continue the conversation with the next customer.
Menu:
burger, frankfurt, sandwich, fries, salad
drinks: coke, sprite, vichy catalan
"""

3.Best system prompt for the task with clear structure and tasks

In [18]:
prompt3 = """
You are OrderBot for "My Dear Frankfurt", a delivery-only fast food restaurant.

Your behavior:
1. Greet the customer warmly
2. Take the full order (food + drinks)
3. Ask about sizes, toppings, and extras
4. Clarify all details
5. Summarize the complete order
6. Ask if anything else is needed
7. Ask for payment

Rules:
- Be friendly and concise
- Always confirm the order before payment
- Never assume missing details (ask!)

Menu:
Burgers: 12.95, 10.00, 7.00
Frankfurt: 10.95, 9.25, 6.50
Sandwich: 11.95, 9.75, 6.75
Fries: 4.50, 3.50
Salad: 7.25

Toppings:
extra cheese 2.00, mushrooms 1.50, martra sausage 3.00,
canadian bacon 3.50, romesco sauce 1.50, peppers 1.00

Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
vichy catalan 5.00
"""

In [23]:

context = [{'role':'system', 'content': prompt1}]

pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'ae1b3eff-9297-4ec4-bb8f-c38c1bd809b2': {'version…

# Prompt Engineering Report – OrderBot Experiment

## Introduction

In this experiment, we explored different prompt engineering techniques to control the behavior of an AI assistant acting as an “OrderBot” for a fast-food restaurant. The goal was to compare how different prompt styles influence the quality, consistency, and reliability of the generated responses.

---

## Prompt Versions Tested

### 1. Zero-shot Prompt

**Description:**
A minimal instruction without examples or detailed structure.

**Prompt:**
“Take a food order from a customer in a friendly way. Ask for the full order, confirm it, and proceed to payment. Menu includes burgers, fries, sandwiches, and drinks.”

**Observation:**
This version worked but was inconsistent. The AI sometimes skipped steps such as confirming the order or asking for drink sizes. It also occasionally invented details that were not provided in the menu (hallucination).

---

### 2. Few-shot Prompt

**Description:**
This version included example conversations to guide the model.

**Prompt (simplified):**
Included a short dialogue showing how the assistant greets, asks for order details, confirms, and finalizes payment.

**Observation:**
This version produced more consistent and structured conversations. The AI followed the pattern shown in the examples, maintained a friendly tone, and rarely skipped steps. However, it was still limited if the examples did not cover all edge cases.

---

### 3. Structured Instruction Prompt (Context-based)

**Description:**
A detailed system prompt with explicit role, steps, rules, and full menu.

**Prompt:**
Defined the assistant as “OrderBot,” included step-by-step behavior (greet, collect order, clarify, summarize, confirm, payment), and provided full menu with prices and options.

**Observation:**
This was the most reliable version. The AI consistently followed all steps, asked clarifying questions, and stayed within the provided menu. It reduced hallucinations significantly and produced predictable, high-quality responses.

---

## Comparison and Issues

* The **zero-shot prompt** was the weakest: it lacked control and led to inconsistent behavior.
* The **few-shot prompt** improved structure but depended heavily on the quality of examples.
* The **structured instruction prompt** performed best, offering strong control and clarity.

Some issues observed:

* Occasional hallucination in simpler prompts (inventing items or skipping steps)
* Missing confirmations or incomplete orders in zero-shot
* Limited flexibility when examples in few-shot were too narrow

---

## What We Learned

Through this experiment, we learned that prompt design has a major impact on AI performance.

* Providing **clear structure and rules** greatly improves reliability.
* **Few-shot examples** help guide tone and format but are not always sufficient on their own.
* **Detailed context prompts** offer the best control and reduce errors such as hallucinations.
* Simpler prompts may work for basic tasks but are not suitable for complex, multi-step interactions.

Overall, we learned that effective prompt engineering is about balancing clarity, structure, and guidance to achieve consistent and accurate results.

---

## Conclusion

The experiment shows that more structured and detailed prompts lead to better AI performance. While zero-shot prompting is quick and simple, it lacks reliability. Few-shot prompting improves consistency, but the best results are achieved with a well-designed instruction-based prompt that clearly defines behavior and constraints.
